# Render Bundle/No-bundle sobre la segmentacion original (v1)

**Objetivo:** volver a la segmentacion binaria original (`seg_binary`, el volumen grueso de
actina) y colorear segun la clasificacion bundle/no-bundle que ya calculamos en el esqueleto,
para una representacion visual clara (ej. para figuras).

**Paso 1 (este notebook, parte principal):** mostrar la segmentacion original como contexto
semi-transparente + el esqueleto coloreado por bundle/no-bundle encima. Esto usa directamente
la clasificacion ya calculada, sin necesidad de propagarla al grosor completo.

**Paso 2 (opcional, al final):** si despues de ver el resultado quieres que el VOLUMEN GRUESO
completo (no solo el esqueleto delgado) este coloreado, hay una seccion aparte para eso -- cada
voxel de la segmentacion toma el color del punto de esqueleto mas cercano (nearest neighbor).

## 1. Carga

In [57]:
import numpy as np
import pandas as pd
from pathlib import Path
import mrcfile
import napari

output_dir = Path(r"C:\PhD\Actin_ET\Bundle Analysis\Amoeboid\L6_P6")
input_file = r"C:\PhD\Actin_ET\Segmentation\Amoeboid\L6_P6\L6_Position_6___actin [1].mrc"
VOXEL_SIZE_A = 12.4

# Segmentacion original (volumen grueso)
with mrcfile.open(input_file, permissive=True) as mrc:
    seg_raw = mrc.data.copy()
seg_binary = (seg_raw > 0).astype(bool)
print(f"Segmentacion original cargada: shape={seg_binary.shape}, voxels={seg_binary.sum():,}")

# Tabla de clasificacion bundle/no-bundle (la que genero el notebook de Bundle Scoring)
scoring_csv_path = output_dir / "Pos16_actin_bundle_scoring.csv"
df_scoring = pd.read_csv(scoring_csv_path)
print(f"Tabla de scoring cargada: {len(df_scoring):,} puntos")
print(f"  Columnas: {list(df_scoring.columns)}")

Segmentacion original cargada: shape=(300, 1024, 1024), voxels=122,088
Tabla de scoring cargada: 24,043 puntos
  Columnas: ['z_voxel', 'y_voxel', 'x_voxel', 'fragment_id', 'orientation_z', 'orientation_y', 'orientation_x', 'n_parallel_neighbors', 'is_bundle']


## 2. Reconstruir el esqueleto coloreado por clasificacion

A partir de la tabla (`z_voxel`, `y_voxel`, `x_voxel`, `is_bundle`), reconstruimos un volumen
donde:
- `0` = fondo (no esqueleto)
- `1` = punto de esqueleto clasificado como NO-bundle
- `2` = punto de esqueleto clasificado como BUNDLE
- `3` = punto de esqueleto SIN clasificar (orientacion no valida, fragmento muy corto)

In [58]:
classification_volume = np.zeros(seg_binary.shape, dtype=np.uint8)

zs = df_scoring["z_voxel"].values
ys = df_scoring["y_voxel"].values
xs = df_scoring["x_voxel"].values
is_bundle = df_scoring["is_bundle"].values.astype(bool)
has_orientation = df_scoring["orientation_z"].notna().values

# Por defecto: sin clasificar (3)
labels_for_points = np.full(len(df_scoring), 3, dtype=np.uint8)
labels_for_points[has_orientation & ~is_bundle] = 1  # no-bundle
labels_for_points[has_orientation & is_bundle] = 2   # bundle

classification_volume[zs, ys, xs] = labels_for_points

n_bundle = (labels_for_points == 2).sum()
n_no_bundle = (labels_for_points == 1).sum()
n_unclassified = (labels_for_points == 3).sum()
print(f"Puntos bundle: {n_bundle:,}")
print(f"Puntos no-bundle: {n_no_bundle:,}")
print(f"Puntos sin clasificar: {n_unclassified:,}")

Puntos bundle: 11,622
Puntos no-bundle: 12,419
Puntos sin clasificar: 2


## 3. Visualizar: segmentacion original (contexto) + esqueleto coloreado

`seg_binary` se muestra semi-transparente como contexto del grosor real del filamento;
`classification_volume` encima, coloreado por categoria.

In [59]:
viewer = napari.Viewer(title="Bundle/No-bundle sobre segmentacion original")

viewer.add_labels(
    seg_binary.astype(np.uint8),
    name="segmentacion original (contexto)",
    opacity=0.25,
)

classification_layer = viewer.add_labels(
    classification_volume,
    name="clasificacion (1=no-bundle, 2=bundle, 3=sin clasificar)",
    opacity=1.0,
)

# Colores explicitos asignados DESPUES de crear la capa -- mas compatible entre versiones de
# napari que pasar 'color'/'color_map' como argumento de add_labels (el nombre del parametro
# y su disponibilidad han cambiado entre versiones).
try:
    # Versiones recientes de napari: color_map es un dict {label: color}
    classification_layer.color_map = {1: "red", 2: "cyan", 3: "gray"}
except AttributeError:
    try:
        # Versiones mas antiguas: el atributo se llama 'color'
        classification_layer.color = {1: "red", 2: "cyan", 3: "gray"}
    except AttributeError:
        print("No se pudo asignar color personalizado en esta version de napari -- "
              "se usara la paleta de colores por defecto (los labels 1/2/3 seguiran "
              "siendo visualmente distintos, solo no coincidiran con rojo/cyan/gris).")

viewer.dims.ndisplay = 3
print("Cyan = bundle, rojo = no-bundle, gris = sin clasificar (fragmento muy corto).")
print("La segmentacion original aparece semi-transparente como contexto del grosor real.")

Cyan = bundle, rojo = no-bundle, gris = sin clasificar (fragmento muy corto).
La segmentacion original aparece semi-transparente como contexto del grosor real.


## 4. Guardar volumen de clasificacion (para reabrir despues sin recalcular)

In [60]:
render_output_path = output_dir / "Pos16_actin_bundle_render.mrc"

with mrcfile.new(str(render_output_path), overwrite=True) as mrc_out:
    mrc_out.set_data(classification_volume.astype(np.int8))
    mrc_out.voxel_size = VOXEL_SIZE_A

print(f"Guardado: {render_output_path}")

Guardado: C:\PhD\Actin_ET\Bundle Analysis\Amoeboid\L6_P6\Pos16_actin_bundle_render.mrc


## 5. (Opcional) Rellenar el GROSOR COMPLETO del filamento, no solo el esqueleto

Si despues de ver el resultado en la seccion 3 quieres que el volumen GRUESO de la segmentacion
original este coloreado (no solo la linea fina del esqueleto), esta seccion propaga la
clasificacion de cada punto del esqueleto a todos los voxels de `seg_binary` que tiene mas cerca,
usando un KDTree de nearest-neighbor.

**Nota:** esto puede tardar mas que las secciones anteriores, dependiendo del numero de voxels
en `seg_binary` (tipicamente mucho mayor que el numero de puntos del esqueleto). Corre solo si
la necesitas.

In [61]:
from scipy.spatial import cKDTree

# Construir KDTree con las posiciones de los puntos de esqueleto YA clasificados (bundle o no-bundle,
# excluyendo los sin clasificar para no propagar incertidumbre)
classified_mask = has_orientation
skeleton_points = np.column_stack([zs[classified_mask], ys[classified_mask], xs[classified_mask]])
skeleton_labels = labels_for_points[classified_mask]  # 1 o 2

print(f"Construyendo KDTree con {len(skeleton_points):,} puntos de esqueleto clasificados...")
tree = cKDTree(skeleton_points)

# Todos los voxels de la segmentacion original
seg_zs, seg_ys, seg_xs = np.where(seg_binary)
seg_points = np.column_stack([seg_zs, seg_ys, seg_xs])
print(f"Propagando a {len(seg_points):,} voxels de la segmentacion original...")

_, nearest_idx = tree.query(seg_points, k=1)
nearest_labels = skeleton_labels[nearest_idx]

filled_classification_volume = np.zeros(seg_binary.shape, dtype=np.uint8)
filled_classification_volume[seg_zs, seg_ys, seg_xs] = nearest_labels

n_bundle_filled = (filled_classification_volume == 2).sum()
n_no_bundle_filled = (filled_classification_volume == 1).sum()
print(f"\nVoxels bundle (volumen completo): {n_bundle_filled:,} "
      f"({100*n_bundle_filled/seg_binary.sum():.1f}%)")
print(f"Voxels no-bundle (volumen completo): {n_no_bundle_filled:,} "
      f"({100*n_no_bundle_filled/seg_binary.sum():.1f}%)")

Construyendo KDTree con 24,041 puntos de esqueleto clasificados...
Propagando a 122,088 voxels de la segmentacion original...

Voxels bundle (volumen completo): 57,233 (46.9%)
Voxels no-bundle (volumen completo): 64,855 (53.1%)


In [62]:
viewer_filled = napari.Viewer(title="Bundle/No-bundle -- VOLUMEN COMPLETO (grosor real)")

filled_layer = viewer_filled.add_labels(
    filled_classification_volume,
    name="clasificacion (volumen completo)",
    opacity=0.9,
)

try:
    filled_layer.color_map = {1: "red", 2: "cyan"}
except AttributeError:
    try:
        filled_layer.color = {1: "red", 2: "cyan"}
    except AttributeError:
        print("No se pudo asignar color personalizado en esta version de napari.")

viewer_filled.dims.ndisplay = 3

print("Cyan = bundle, rojo = no-bundle. Ahora cada voxel del filamento (no solo el esqueleto)")
print("esta coloreado segun la clasificacion del punto de esqueleto mas cercano.")

Cyan = bundle, rojo = no-bundle. Ahora cada voxel del filamento (no solo el esqueleto)
esta coloreado segun la clasificacion del punto de esqueleto mas cercano.


In [63]:
filled_output_path = output_dir / "Pos16_actin_bundle_render_filled.mrc"

with mrcfile.new(str(filled_output_path), overwrite=True) as mrc_out:
    mrc_out.set_data(filled_classification_volume.astype(np.int8))
    mrc_out.voxel_size = VOXEL_SIZE_A

print(f"Guardado: {filled_output_path}")

Guardado: C:\PhD\Actin_ET\Bundle Analysis\Amoeboid\L6_P6\Pos16_actin_bundle_render_filled.mrc
